In [0]:
%pip install statsmodels
dbutils.library.restartPython()

# 05: Business Recommendation

**Question for the marketing team:** Which acquisition channel deserves more budget?

**What the analysis found:**
- Paid search sellers are about 17 percentage points more likely to start selling than comparable organic search sellers (notebook 03).
- Once active, sellers perform the same regardless of channel (notebooks 02 and 03).
- Seller characteristics predict activation better than channel does (notebook 04).

**The limitation:** the dataset contains no marketing costs. So instead of calculating return on investment directly, this notebook calculates **break-even thresholds** that the marketing team can compare against their actual cost per acquired seller.

In [0]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportion_confint

df = spark.table("workspace.marts.fact_seller_channel_performance").toPandas()
df["is_active"] = df["is_active_90d"].astype(int)
df["revenue_90d"] = df["revenue_90d"].astype(float)
for col in ["lead_type", "business_type"]:
    df[col] = df[col].fillna("unknown")

rng = np.random.default_rng(42)

paid = df.loc[df["channel_group"] == "paid_search", "is_active"].values
organic = df.loc[df["channel_group"] == "organic_search", "is_active"].values
print(f"Paid search: {len(paid)} sellers, {paid.mean():.1%} active")
print(f"Organic search: {len(organic)} sellers, {organic.mean():.1%} active")

In [0]:
ratio = paid.mean() / organic.mean()

boot_ratios = [
    rng.choice(paid, len(paid)).mean() / rng.choice(organic, len(organic)).mean()
    for _ in range(5000)
]
ratio_low, ratio_high = np.percentile(boot_ratios, [2.5, 97.5])

print(f"Paid search activation is {ratio:.2f}x organic search (95% CI: {ratio_low:.2f}x to {ratio_high:.2f}x)")
print(f"\nBreak-even: paid search can cost up to {ratio - 1:.0%} more per acquired seller than organic")
print(f"and still deliver the same cost per ACTIVE seller (95% CI: {ratio_low - 1:.0%} to {ratio_high - 1:.0%}).\n")

premiums = [0.0, 0.2, 0.4, 0.6, 0.8]
scenarios = pd.DataFrame({
    "paid_cost_premium": [f"{p:.0%}" for p in premiums],
    "paid_cost_per_active_vs_organic": [(1 + p) / ratio for p in premiums],
})
scenarios["verdict"] = np.where(
    scenarios["paid_cost_per_active_vs_organic"] < 1, "Paid search cheaper per active seller",
    "Organic cheaper per active seller")
scenarios["paid_cost_per_active_vs_organic"] = scenarios["paid_cost_per_active_vs_organic"].round(2)
scenarios

In [0]:
effect = 0.167          # marginal effect from notebook 03 (with controls)
effect_low, effect_high = 0.067, 0.267

shift = pd.DataFrame({"sellers_acquired_via_paid_instead_of_organic": [50, 100, 200]})
shift["extra_active_sellers"] = (shift.iloc[:, 0] * effect).round(0)
shift["range_low"] = (shift.iloc[:, 0] * effect_low).round(0)
shift["range_high"] = (shift.iloc[:, 0] * effect_high).round(0)
shift

In [0]:
def activation_by(col, min_sellers=20):
    rows = []
    for value, group in df.groupby(col):
        n, k = len(group), int(group["is_active"].sum())
        if n < min_sellers:
            continue
        low, high = proportion_confint(k, n, method="wilson")
        rows.append({col: value, "sellers": n, "activation_rate": k / n, "ci_low": low, "ci_high": high})
    return pd.DataFrame(rows).sort_values("activation_rate", ascending=False).round(3)

print("Activation by lead type:")
display(activation_by("lead_type"))
print("\nActivation by business type:")
display(activation_by("business_type"))

## Recommendation

**1. Favor paid search, within a cost limit.**
Paid search sellers activate 1.41 times as often as organic search sellers (95% CI: 1.12x to 1.78x), an effect that holds after controlling for seller characteristics and sales rep. Paid search can therefore cost up to about 41% more per acquired seller and still match organic search on cost per active seller. Because the confidence interval is wide (break-even between 12% and 78%), paid search is clearly worthwhile only if its cost premium is below about 12%; between 12% and 78%, the answer depends on actual costs and should be tested.

**2. Put more effort into lead qualification than into channel mix.**
Lead type is a stronger driver of activation than channel: large online sellers activate at 62% versus 25% for offline businesses, and resellers at 49% versus 32% for manufacturers. Prioritizing high-activation lead types in sales outreach, and offering extra onboarding support to offline businesses and manufacturers, is likely to raise activation more than reallocating channel budget.

**3. Do not expect channel to change revenue per seller.**
Once active, sellers from every channel perform the same on revenue, orders, reviews, and delivery. Channel budget decisions should be judged on activation and cost, not on expected seller quality.

**4. Fix channel tracking.**
About a quarter of acquired sellers (193 of 842) have no recorded channel. Better attribution would sharpen every estimate in this analysis.

**5. Validate with an experiment.**
This analysis is observational: sellers chose their own channels, so unmeasured differences between paid and organic leads could explain part of the effect. A controlled budget test (for example, increasing paid search spend in selected regions or time periods and comparing activation against a control group) would confirm whether the effect is causal before a large budget shift.
